In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import cv2
# 1. 加载对齐矩阵 M
# 注意：官方的 CSV 通常没有表头，或者第一行就是数据
mat_df = pd.read_csv("Xenium_Prime_Human_Prostate_FFPE_he_imagealignment.csv", header=None)
M = mat_df.values

# 2. 计算逆矩阵 M_inv
M_inv = np.linalg.inv(M)

# 3. 加载 Xenium 细胞核边界数据 (微米单位)
# 这里以 nucleus_boundaries.csv.gz 为例
boundaries = pd.read_csv("nucleus_boundaries.csv.gz")


In [2]:
boundaries

,cell_id,vertex_x,vertex_y,label_id
0,aaaamcnn-1,141.3125,2475.2000,1
1,aaaamcnn-1,140.6750,2475.4126,1
2,aaaamcnn-1,138.9750,2476.2625,1
3,aaaamcnn-1,138.1250,2477.1125,1
4,aaaamcnn-1,137.7000,2478.1750,1
...,...,...,...,...
4695626,oiohaagd-1,4880.7000,3249.5500,191544
4695627,oiohaagd-1,4880.2750,3249.3376,191544
4695628,oiohaagd-1,4879.8500,3249.3376,191544
4695629,oiohaagd-1,4879.6377,3249.1250,191544


In [3]:
# 4. 执行转换步骤
# 第一步：微米 -> Xenium 内部像素 (除以 0.2125)
boundaries['x_xe_px'] = boundaries['vertex_x'] / 0.2125
boundaries['y_xe_px'] = boundaries['vertex_y'] / 0.2125

# 第二步：构建齐次坐标矩阵 [X, Y, 1]
xe_pixels = boundaries[['x_xe_px', 'y_xe_px']].values
ones = np.ones((xe_pixels.shape[0], 1))
xe_homogeneous = np.hstack([xe_pixels, ones]) # 变成 [N, 3] 矩阵

In [4]:
he_pixels_homogeneous = (M_inv @ xe_homogeneous.T).T

# 5. 将结果保存回 DataFrame
boundaries['he_pixel_x'] = he_pixels_homogeneous[:, 0]
boundaries['he_pixel_y'] = he_pixels_homogeneous[:, 1]

# 查看前几行结果
print(boundaries[['cell_id', 'vertex_x', 'vertex_y', 'he_pixel_x', 'he_pixel_y']].head())

      cell_id  vertex_x   vertex_y    he_pixel_x   he_pixel_y
0  aaaamcnn-1  141.3125  2475.2000  15134.046213  6447.093366
1  aaaamcnn-1  140.6750  2475.4126  15133.246185  6444.772455
2  aaaamcnn-1  138.9750  2476.2625  15130.079127  6438.593756
3  aaaamcnn-1  138.1250  2477.1125  15126.942935  6435.520024
4  aaaamcnn-1  137.7000  2478.1750  15123.046117  6434.006580


In [5]:
# 1. 创建一个新的 DataFrame 用于保存，避免直接破坏原始内存中的 boundaries 数据
output_df = boundaries.copy()

# 2. 删除原始的微米坐标列 (vertex_x, vertex_y)
output_df = output_df.drop(columns=['x_xe_px','y_xe_px','vertex_x', 'vertex_y'])

# 3. 将 H&E 像素坐标重命名为原始列名
output_df = output_df.rename(columns={
    'he_pixel_x': 'vertex_x',
    'he_pixel_y': 'vertex_y'
})

# 4. 调整列顺序，使其与原始文件完全一致 (可选，但推荐)
# 假设原始列顺序是 [cell_id, vertex_x, vertex_y, ...]
# 获取除 vertex 之外的所有列名，重新排列
cols = list(output_df.columns)

In [6]:
# 定义新文件名
output_filename = "nucleus_boundaries_HE_coords.csv.gz"

print(f"正在保存转换后的坐标至: {output_filename} ...")

# 保存为 .csv.gz 格式
# index=False 确保不把行索引存进去
output_df.to_csv(output_filename, index=False, compression='gzip')

print("保存完成！")


正在保存转换后的坐标至: nucleus_boundaries_HE_coords.csv.gz ...
保存完成！


In [7]:
he_image_path = "/sibcb1/chenluonanlab8/zoujiawei/Prostate/Xenium_Prime_Human_Prostate_FFPE_he_image.ome.tif"

In [ ]:
import tifffile
import zarr
import matplotlib.pyplot as plt
import numpy as np

# --- 1. 准备数据 (假设 boundaries 已经算好了 he_pixel_x/y) ---
df = boundaries 
sample_idx = 5000
center_x = int(df.iloc[sample_idx]['he_pixel_x'])
center_y = int(df.iloc[sample_idx]['he_pixel_y'])

crop_size = 500
x_s = max(0, center_x - crop_size // 2)
y_s = max(0, center_y - crop_size // 2)

# --- 2. 使用 Zarr 方式打开图片 (非常高效) ---
# aszarr=True 会创建一个轻量级的映射，不加载实际图像数据
store = tifffile.imread(he_image_path, aszarr=True)
z_img = zarr.open(store, mode='r')

# Xenium 的 OME-TIFF 通常是一个金字塔结构
# z_img[0] 是最高分辨率层
# 它的形状通常是 (Height, Width, Channels) 或者 (Channels, Height, Width)
full_img = z_img[0]
print(f"完整图片形状: {full_img.shape}")

# --- 3. 切片读取 ROI ---
# 自动处理维度问题 (针对 H&E, 常见是 Y, X, C)
if full_img.ndim == 3:
    # 假设形状是 (H, W, 3)
    roi = full_img[y_s : y_s + crop_size, x_s : x_s + crop_size, :]
else:
    # 如果只有 (H, W)
    roi = full_img[y_s : y_s + crop_size, x_s : x_s + crop_size]

# --- 4. 筛选落在该区域内的细胞点 ---
mask = (df['he_pixel_x'] >= x_s) & (df['he_pixel_x'] <= x_s + crop_size) & \
       (df['he_pixel_y'] >= y_s) & (df['he_pixel_y'] <= y_s + crop_size)
df_plot = df[mask].copy()

# 转换为相对于 ROI 的局部坐标
df_plot['plot_x'] = df_plot['he_pixel_x'] - x_s
df_plot['plot_y'] = df_plot['he_pixel_y'] - y_s

# --- 5. 可视化 ---
plt.figure(figsize=(10, 10))

# 检查数据类型，如果是 uint16 需要缩放显示
img_to_show = np.array(roi) # 转为普通 numpy 数组方便显示
if img_to_show.dtype == np.uint16:
    img_to_show = (img_to_show / (img_to_show.max() / 255)).astype(np.uint8)

plt.imshow(img_to_show)

# 绘制红点
plt.scatter(df_plot['plot_x'], df_plot['plot_y'], 
            s=15, c='red', alpha=0.8, edgecolors='white', linewidths=0.5)

plt.title(f"H&E Alignment (Zarr Mode) Center: {center_x}, {center_y}")
plt.axis('off')
plt.show()
